# LocaleLive — LangSmith Trace Explorer

Uses the official **LangSmith SDK** (`langsmith.Client`) to pull LangGraph execution traces and explore the full agent flow for every query.

**What you get**
- List of LangSmith projects linked to this deployment
- All runs (traces) with token usage, latency, errors
- Per-node LangGraph breakdown: which agent ran, how long, input/output
- Full trace replay for a single request
- Token cost estimation per agent
- Error run deep-dive

**Log group for cross-referencing:** `/aws/lambda/localelive-backend`

---
Run the cell below first to install dependencies, then restart the kernel.

In [ ]:
%pip install langsmith pandas matplotlib seaborn python-dotenv --quiet

In [ ]:
import os
import json
from datetime import datetime, timezone, timedelta
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from dotenv import load_dotenv
from langsmith import Client

load_dotenv(Path("credentials.env"))

LANGCHAIN_API_KEY = os.environ["LANGCHAIN_API_KEY"]

client = Client(api_key=LANGCHAIN_API_KEY)
print(f"Connected to LangSmith — workspace: {client.workspace_id}")

## 1 — List projects

In [ ]:
projects = list(client.list_projects())
proj_df = pd.DataFrame([
    {
        "name":        p.name,
        "id":          str(p.id),
        "run_count":   p.run_count,
        "created_at":  p.created_at,
    }
    for p in projects
])
print(f"{len(proj_df)} project(s)")
proj_df

In [ ]:
# ── Pick the project to analyse ───────────────────────────────────────────────
# Set PROJECT_NAME to any name from the table above, or leave as None to use
# the first (most recently created) project.
PROJECT_NAME = None

if PROJECT_NAME is None and not proj_df.empty:
    PROJECT_NAME = proj_df.iloc[0]["name"]

# ── Time window ───────────────────────────────────────────────────────────────
LOOKBACK_HOURS = 72
start_dt = datetime.now(timezone.utc) - timedelta(hours=LOOKBACK_HOURS)

print(f"Project : {PROJECT_NAME}")
print(f"Window  : last {LOOKBACK_HOURS}h (from {start_dt.strftime('%Y-%m-%d %H:%M UTC')})")

## 2 — Pull all top-level runs (one per user query)

Top-level runs are the root of each LangGraph trace — one per `PUT /query` call.

In [ ]:
MAX_RUNS = 2000

raw_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=start_dt,
    is_root=True,           # top-level traces only
    limit=MAX_RUNS,
))
print(f"Fetched {len(raw_runs)} root runs")

def _ms(run, attr):
    """Duration in seconds between two datetime attrs, or None."""
    a = getattr(run, attr, None)
    b = getattr(run, "end_time", None)
    s = getattr(run, "start_time", None)
    if attr == "duration" and s and b:
        return (b - s).total_seconds()
    return None

records = []
for r in raw_runs:
    dur = (r.end_time - r.start_time).total_seconds() if r.end_time and r.start_time else None
    tok = r.total_tokens or 0
    records.append({
        "run_id":          str(r.id),
        "name":            r.name,
        "status":          r.status,
        "start_time":      r.start_time,
        "end_time":        r.end_time,
        "duration_s":      dur,
        "total_tokens":    tok,
        "prompt_tokens":   r.prompt_tokens or 0,
        "completion_tokens": r.completion_tokens or 0,
        "error":           r.error,
    })

runs_df = pd.DataFrame(records)
if not runs_df.empty:
    runs_df = runs_df.sort_values("start_time").reset_index(drop=True)
    print(runs_df["status"].value_counts().to_string())
    print(f"\nDuration — p50: {runs_df['duration_s'].quantile(0.5):.2f}s  "
          f"p95: {runs_df['duration_s'].quantile(0.95):.2f}s")
    runs_df.tail(5)

## 3 — Latency & token usage overview

In [ ]:
if not runs_df.empty and runs_df["duration_s"].notna().sum() >= 2:
    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    # Latency histogram
    d = runs_df["duration_s"].dropna()
    axes[0].hist(d, bins=30, color="steelblue", edgecolor="white")
    for pct, color in [(0.5, "lime"), (0.95, "orange"), (0.99, "red")]:
        v = d.quantile(pct)
        axes[0].axvline(v, linestyle="--", color=color, linewidth=1.5,
                        label=f"p{int(pct*100)}={v:.1f}s")
    axes[0].set_title("End-to-end latency")
    axes[0].set_xlabel("Duration (s)")
    axes[0].set_ylabel("Count")
    axes[0].legend(fontsize=8)

    # Total tokens histogram
    t = runs_df["total_tokens"].dropna()
    axes[1].hist(t, bins=30, color="seagreen", edgecolor="white")
    axes[1].set_title("Total tokens per query")
    axes[1].set_xlabel("Tokens")
    axes[1].set_ylabel("Count")

    # Success vs error
    status_counts = runs_df["status"].value_counts()
    axes[2].pie(status_counts, labels=status_counts.index, autopct="%1.0f%%",
                colors=["steelblue", "crimson", "orange"][:len(status_counts)])
    axes[2].set_title("Run status")

    plt.tight_layout()
    plt.show()

## 4 — Pull all child runs (individual LangGraph nodes)

Each root trace contains child runs — one per agent node that executed. This gives per-agent latency and token usage.

In [ ]:
# Fetch all runs (root + children) in the window, filter to LLM and chain types
all_runs = list(client.list_runs(
    project_name=PROJECT_NAME,
    start_time=start_dt,
    limit=10_000,
))
print(f"Total runs (all depths): {len(all_runs)}")

node_records = []
for r in all_runs:
    dur = (r.end_time - r.start_time).total_seconds() if r.end_time and r.start_time else None
    node_records.append({
        "run_id":       str(r.id),
        "parent_id":    str(r.parent_run_id) if r.parent_run_id else None,
        "name":         r.name,
        "run_type":     r.run_type,
        "status":       r.status,
        "start_time":   r.start_time,
        "duration_s":   dur,
        "total_tokens": r.total_tokens or 0,
        "prompt_tokens":     r.prompt_tokens or 0,
        "completion_tokens": r.completion_tokens or 0,
        "error":        r.error,
    })

nodes_df = pd.DataFrame(node_records).sort_values("start_time").reset_index(drop=True)
print(nodes_df["run_type"].value_counts().to_string())
print()
print("Top node names:")
print(nodes_df["name"].value_counts().head(15).to_string())

## 5 — Per-agent latency and token breakdown

In [ ]:
# Focus on the known LocaleLive agent node names
AGENT_NODES = [
    "assistant_agent", "IoT_engine", "GoogleMaps",
    "scrapper", "generator_agent", "reviewer_agent", "finalize_turn"
]

agent_df = nodes_df[nodes_df["name"].isin(AGENT_NODES)].copy()
print(f"{len(agent_df)} agent-node runs")

if not agent_df.empty:
    stats = (
        agent_df.groupby("name")
        .agg(
            count=("run_id", "count"),
            p50_s=("duration_s", lambda x: x.quantile(0.5)),
            p95_s=("duration_s", lambda x: x.quantile(0.95)),
            avg_tokens=("total_tokens", "mean"),
            total_tokens=("total_tokens", "sum"),
            errors=("error", lambda x: x.notna().sum()),
        )
        .round(2)
        .reindex([n for n in AGENT_NODES if n in agent_df["name"].values])
    )
    display(stats)

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    # Latency box plot per agent
    order = agent_df.groupby("name")["duration_s"].median().sort_values().index.tolist()
    sns.boxplot(data=agent_df, x="name", y="duration_s", order=order, ax=axes[0])
    axes[0].set_title("Latency per agent node")
    axes[0].set_xlabel("")
    axes[0].set_ylabel("Duration (s)")
    axes[0].tick_params(axis="x", rotation=20)

    # Avg token usage per agent
    tok_order = stats["avg_tokens"].sort_values(ascending=False).index.tolist()
    stats.loc[tok_order, "avg_tokens"].plot(kind="bar", ax=axes[1], color="seagreen")
    axes[1].set_title("Avg tokens per agent node")
    axes[1].set_xlabel("")
    axes[1].set_ylabel("Tokens")
    axes[1].tick_params(axis="x", rotation=20)

    plt.tight_layout()
    plt.show()

## 6 — LangGraph flow: routing path frequency

Reconstructs which agent sequence ran for each root trace.

In [ ]:
if not nodes_df.empty and not runs_df.empty:
    root_ids = set(runs_df["run_id"])

    # Map each agent run to its root trace via parent_id
    # Build a parent→root lookup (runs may be 2-3 levels deep)
    id_to_parent = {r["run_id"]: r["parent_id"] for _, r in nodes_df.iterrows()}

    def find_root(run_id, depth=0):
        if depth > 10 or run_id is None:
            return None
        if run_id in root_ids:
            return run_id
        return find_root(id_to_parent.get(run_id), depth + 1)

    agent_df2 = agent_df.copy()
    agent_df2["root_id"] = agent_df2["run_id"].apply(find_root)

    # Build ordered path per root trace
    paths = (
        agent_df2.dropna(subset=["root_id"])
        .sort_values("start_time")
        .groupby("root_id")["name"]
        .apply(lambda names: " → ".join(names))
        .value_counts()
        .reset_index()
    )
    paths.columns = ["path", "count"]
    paths["pct"] = (paths["count"] / paths["count"].sum() * 100).round(1)
    print(f"{len(paths)} distinct routing paths")
    display(paths.head(15))

    if len(paths) <= 10:
        fig, ax = plt.subplots(figsize=(10, max(3, len(paths) * 0.5)))
        paths.set_index("path")["count"].sort_values().plot(
            kind="barh", ax=ax, color="steelblue")
        ax.set_title("LangGraph routing path frequency")
        ax.set_xlabel("Count")
        plt.tight_layout()
        plt.show()

## 7 — Token usage over time

In [ ]:
if not runs_df.empty and len(runs_df) >= 3:
    ts = runs_df.set_index("start_time").sort_index()
    hourly = ts[["total_tokens", "prompt_tokens", "completion_tokens"]].resample("1h").sum()

    fig, axes = plt.subplots(1, 2, figsize=(14, 4))

    hourly[["prompt_tokens", "completion_tokens"]].plot(
        kind="area", stacked=True, ax=axes[0], alpha=0.75,
        color=["#aec7e8", "#1f77b4"]
    )
    axes[0].set_title("Token usage per hour (stacked)")
    axes[0].set_xlabel("UTC")
    axes[0].set_ylabel("Tokens")
    axes[0].xaxis.set_major_formatter(mdates.DateFormatter("%m-%d %Hh"))
    plt.setp(axes[0].xaxis.get_majorticklabels(), rotation=30)

    # Scatter: latency vs tokens per query
    axes[1].scatter(runs_df["total_tokens"], runs_df["duration_s"],
                    alpha=0.5, s=25, color="seagreen")
    axes[1].set_title("Latency vs token count per query")
    axes[1].set_xlabel("Total tokens")
    axes[1].set_ylabel("Duration (s)")

    plt.tight_layout()
    plt.show()

## 8 — Error run deep-dive

In [ ]:
error_runs = runs_df[runs_df["error"].notna()].copy()
print(f"Error runs: {len(error_runs)} of {len(runs_df)}")

if not error_runs.empty:
    pd.set_option("display.max_colwidth", 300)
    display(error_runs[["start_time", "run_id", "name", "duration_s", "error"]]
            .reset_index(drop=True))

## 9 — Full trace replay for a single run

Shows every node's input/output and timing for one root trace. Useful for understanding exactly what the model saw and returned at each step.

In [ ]:
# Set to any run_id from runs_df, or leave as None to auto-pick the slowest
TRACE_RUN_ID = None

if TRACE_RUN_ID is None and not runs_df.empty:
    TRACE_RUN_ID = runs_df.loc[runs_df["duration_s"].idxmax(), "run_id"]
    print(f"Auto-selected slowest run: {TRACE_RUN_ID}")

if TRACE_RUN_ID:
    # Fetch the root run and all its children
    root = client.read_run(TRACE_RUN_ID)
    children = list(client.list_runs(
        project_name=PROJECT_NAME,
        trace_id=TRACE_RUN_ID,
    ))
    children_sorted = sorted(
        [r for r in children if str(r.id) != TRACE_RUN_ID],
        key=lambda r: r.start_time or datetime.min.replace(tzinfo=timezone.utc)
    )

    dur = (root.end_time - root.start_time).total_seconds() if root.end_time and root.start_time else "?"
    print(f"\nTrace: {TRACE_RUN_ID}")
    print(f"Status: {root.status}  |  Duration: {dur}s  |  Tokens: {root.total_tokens}")
    if root.error:
        print(f"Error: {root.error}")
    print(f"\n{'Node':<22} {'Type':<10} {'Status':<10} {'Duration':>10}  {'Tokens':>8}")
    print("-" * 70)
    for r in children_sorted:
        node_dur = (
            f"{(r.end_time - r.start_time).total_seconds():.2f}s"
            if r.end_time and r.start_time else "—"
        )
        print(f"{r.name:<22} {r.run_type:<10} {r.status:<10} {node_dur:>10}  {r.total_tokens or 0:>8}")
    print("\n── Root run input ──")
    print(json.dumps(root.inputs, indent=2, default=str)[:1000])
    print("\n── Root run output ──")
    print(json.dumps(root.outputs, indent=2, default=str)[:1000])

## 10 — Inspect a single node's LLM input/output

Drill into what the model was actually sent and what it replied.

In [ ]:
# Change to any run_id from the trace above (pick a child node)
NODE_RUN_ID = None

if NODE_RUN_ID is None and TRACE_RUN_ID:
    # Auto-pick the first LLM child of the trace
    llm_children = [r for r in children_sorted if r.run_type == "llm"]
    if llm_children:
        NODE_RUN_ID = str(llm_children[0].id)
        print(f"Auto-selected first LLM node: {llm_children[0].name} ({NODE_RUN_ID})")

if NODE_RUN_ID:
    node = client.read_run(NODE_RUN_ID)
    print(f"\nNode: {node.name}  |  Type: {node.run_type}  |  Status: {node.status}")
    print(f"Tokens — prompt: {node.prompt_tokens}  completion: {node.completion_tokens}")
    print("\n── Inputs (messages sent to LLM) ──")
    messages = node.inputs.get("messages") or node.inputs.get("prompts") or []
    for msg in messages:
        role = msg.get("role", msg.get("type", "?"))
        content = msg.get("content", "") or ""
        print(f"  [{role}] {str(content)[:400]}")
    print("\n── Output (LLM reply) ──")
    print(json.dumps(node.outputs, indent=2, default=str)[:1000])

## 11 — Export summary to CSV

In [ ]:
if not runs_df.empty:
    out = Path("localelive_langsmith_runs.csv")
    runs_df.drop(columns=["error"], errors="ignore").to_csv(out, index=False)
    print(f"Exported {len(runs_df)} root runs to {out}")

    summary = {
        "project":           PROJECT_NAME,
        "window_hours":      LOOKBACK_HOURS,
        "total_runs":        len(runs_df),
        "error_rate":        round(runs_df["error"].notna().mean(), 4),
        "p50_latency_s":     round(runs_df["duration_s"].quantile(0.5), 3),
        "p95_latency_s":     round(runs_df["duration_s"].quantile(0.95), 3),
        "total_tokens":      int(runs_df["total_tokens"].sum()),
        "avg_tokens_per_run": round(runs_df["total_tokens"].mean(), 1),
    }
    print(json.dumps(summary, indent=2))